# SupplyMind AI — Model Comparison & Champion Selection

This notebook compares all trained delay-prediction candidates using validation metrics, selects the production champion using an operationally balanced policy, promotes the winning artifact to `models/champion`, and records the final findings.

## Selection policy

The champion is selected by:
1. **Balanced accuracy** — primary metric to avoid majority-class bias.
2. **F1 score** — tie-breaker balancing precision and recall.
3. **ROC-AUC** — final tie-breaker measuring ranking quality.

## 1. Imports and project paths


In [1]:
from __future__ import annotations

import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
REPORT_ROOT = PROJECT_ROOT / "reports"
MODEL_ROOT = PROJECT_ROOT / "models"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPORT_ROOT:", REPORT_ROOT)
print("MODEL_ROOT:", MODEL_ROOT)


PROJECT_ROOT: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai
REPORT_ROOT: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai/reports
MODEL_ROOT: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai/models


## 2. Candidate models


In [2]:
CANDIDATE_NAMES = [
    "logistic_regression",
    "random_forest",
    "xgboost",
    "hist_gradient_boosting",
]

CANDIDATE_NAMES


['logistic_regression', 'random_forest', 'xgboost', 'hist_gradient_boosting']

## 3. Load validation metrics

Each candidate notebook writes `reports/models/<model_name>/validation_metrics.json`. These files are the single source of truth for model comparison.


In [3]:
rows = []

for name in CANDIDATE_NAMES:
    metrics_path = REPORT_ROOT / "models" / name / "validation_metrics.json"

    if not metrics_path.exists():
        print(f"Missing metrics for {name}: {metrics_path}")
        continue

    metrics = json.loads(metrics_path.read_text())
    rows.append({"model_name": name, **metrics})

if not rows:
    raise RuntimeError("No candidate validation metrics were found.")

comparison = pd.DataFrame(rows)

required_columns = {
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "threshold",
}

missing_columns = required_columns - set(comparison.columns)

if missing_columns:
    raise RuntimeError(
        f"Missing metrics: {sorted(missing_columns)}. "
        "Rerun notebooks 07-10 with the updated evaluation module."
    )

comparison


,model_name,accuracy,precision,recall,f1,roc_auc,average_precision,balanced_accuracy,specificity,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,threshold
0,logistic_regression,0.646358,0.673986,0.750186,0.710047,0.740043,0.834546,0.627400,0.504614,4976,4885,3363,10099,0.495386,0.249814,0.34
1,random_forest,0.648244,0.674777,0.753974,0.712181,0.739477,0.835119,0.628939,0.503904,4969,4892,3312,10150,0.496096,0.246026,0.38
2,xgboost,0.655962,0.698786,0.709999,0.704348,0.739064,0.834281,0.646095,0.582192,5741,4120,3904,9558,0.417808,0.290001,0.41
3,hist_gradient_boosting,0.691120,0.881213,0.537290,0.667559,0.739468,0.834411,0.719208,0.901126,8886,975,6229,7233,0.098874,0.462710,0.62


## 4. Rank candidate models


In [4]:
comparison = comparison.sort_values(
    ["balanced_accuracy", "f1", "roc_auc"],
    ascending=[False, False, False],
).reset_index(drop=True)

display_columns = [
    "model_name",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
    "balanced_accuracy",
    "specificity",
    "true_negative",
    "false_positive",
    "false_negative",
    "true_positive",
    "threshold",
]

comparison[display_columns]


,model_name,accuracy,precision,recall,f1,roc_auc,average_precision,balanced_accuracy,specificity,true_negative,false_positive,false_negative,true_positive,threshold
0,hist_gradient_boosting,0.691120,0.881213,0.537290,0.667559,0.739468,0.834411,0.719208,0.901126,8886,975,6229,7233,0.62
1,xgboost,0.655962,0.698786,0.709999,0.704348,0.739064,0.834281,0.646095,0.582192,5741,4120,3904,9558,0.41
2,random_forest,0.648244,0.674777,0.753974,0.712181,0.739477,0.835119,0.628939,0.503904,4969,4892,3312,10150,0.38
3,logistic_regression,0.646358,0.673986,0.750186,0.710047,0.740043,0.834546,0.627400,0.504614,4976,4885,3363,10099,0.34


## 5. Select champion

The first row after sorting is the champion candidate.


In [5]:
champion_name = str(comparison.iloc[0]["model_name"])
champion_threshold = float(comparison.iloc[0]["threshold"])

print("Selected champion:", champion_name)
print("Threshold:", champion_threshold)


Selected champion: hist_gradient_boosting
Threshold: 0.6200000000000003


## 6. Promote champion artifact

The winning candidate artifact is copied from `models/candidates/<model_name>` to `models/champion`.

The previous champion directory is replaced completely so stale artifacts cannot survive promotion.


In [6]:
candidate_dir = MODEL_ROOT / "candidates" / champion_name
champion_dir = MODEL_ROOT / "champion"

candidate_model_path = candidate_dir / "model.joblib"
candidate_metadata_path = candidate_dir / "metadata.json"

if not candidate_model_path.exists():
    raise FileNotFoundError(
        f"Missing candidate model: {candidate_model_path}"
    )

if not candidate_metadata_path.exists():
    raise FileNotFoundError(
        f"Missing candidate metadata: {candidate_metadata_path}"
    )

candidate_metadata = json.loads(
    candidate_metadata_path.read_text()
)

validation_metrics_path = (
    REPORT_ROOT
    / "models"
    / champion_name
    / "validation_metrics.json"
)

validation_metrics = json.loads(
    validation_metrics_path.read_text()
)

if champion_dir.exists():
    shutil.rmtree(champion_dir)

champion_dir.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    candidate_model_path,
    champion_dir / "model.joblib",
)

champion_metadata = {
    **candidate_metadata,
    "model_name": champion_name,
    "model_version": "1.0.0",
    "promoted_at": datetime.now(timezone.utc).isoformat(),
    "threshold": champion_threshold,
    "selection_metric": (
        "balanced_accuracy_then_f1_then_roc_auc"
    ),
    "validation_metrics": validation_metrics,
}

(champion_dir / "metadata.json").write_text(
    json.dumps(
        champion_metadata,
        indent=2,
        default=str,
    )
)

print("Champion promoted successfully.")
print("Champion directory:", champion_dir)


Champion promoted successfully.
Champion directory: /Users/karima/IronHack/Ironhack-challenges/supplymind-ai/models/champion


## 7. Verify promoted champion


In [7]:
promoted_metadata = json.loads(
    (MODEL_ROOT / "champion" / "metadata.json").read_text()
)

model_size_mb = (
    (MODEL_ROOT / "champion" / "model.joblib").stat().st_size
    / (1024 ** 2)
)

print("Champion:", promoted_metadata["model_name"])
print("Threshold:", promoted_metadata["threshold"])
print("Selection metric:", promoted_metadata["selection_metric"])
print(f"Artifact size: {model_size_mb:.2f} MB")

promoted_metadata


Champion: hist_gradient_boosting
Threshold: 0.6200000000000003
Selection metric: balanced_accuracy_then_f1_then_roc_auc
Artifact size: 0.46 MB


{'model_name': 'hist_gradient_boosting',
 'model_version': '1.0.0',
 'threshold': 0.6200000000000003,
 'validation_metrics': {'accuracy': 0.6911203532993183,
  'precision': 0.8812134502923976,
  'recall': 0.5372901500519982,
  'f1': 0.6675588371019843,
  'roc_auc': 0.7394683930132029,
  'average_precision': 0.83441126194855,
  'balanced_accuracy': 0.7192078982690779,
  'specificity': 0.9011256464861576,
  'true_negative': 8886,
  'false_positive': 975,
  'false_negative': 6229,
  'true_positive': 7233,
  'false_positive_rate': 0.09887435351384241,
  'false_negative_rate': 0.4627098499480018,
  'threshold': 0.6200000000000003},
 'promoted_at': '2026-08-13T22:43:17.411233+00:00',
 'selection_metric': 'balanced_accuracy_then_f1_then_roc_auc'}

## 8. Findings

### Model-selection findings

- The earlier recall-first threshold policy produced extremely high false-positive rates and classified almost every shipment as delayed.
- Threshold selection was updated to use operational constraints and balanced accuracy.
- Champion selection now prioritizes **balanced accuracy**, then **F1**, then **ROC-AUC**.
- This provides a more realistic trade-off between detecting delays and avoiding unnecessary alerts.

### Current champion

After the updated evaluation policy, the latest experiment selected **HistGradientBoosting** with a threshold around **0.62**. The exact values printed above remain the source of truth.

### Operational interpretation

The selected model favors higher precision and specificity compared with the original recall-heavy champion. This reduces false alerts and makes predicted delivery risk more useful for operational decision-making.

### Deployment implication

The promoted champion artifact under `models/champion/` should be used by the FastAPI prediction runtime. Before deployment, verify the artifact size and confirm that `metadata.json` references the newly promoted model rather than the previous Random Forest champion.
